# GitSkills Exploration

A playground for poking around the GitSkills SQLite sample. Each section has a short note, then a code cell.
Lines marked `# <- change me` are the ones meant for editing.

- Run with **Shift+Enter**, or **Run All** at the top.
- Run section 1 first. Section 4 needs section 3, and section 6 needs section 4.
- If you restart the kernel, run section 1 again (and 3 → 4 before 6).

## 1. Connect

This opens `data/sample/agent_skills_sample.db` in **read-only** mode, so nothing we run can change the data.
Every other cell talks to the database through this connection, so run this cell first.
It also defines a small helper, `sql("SELECT ...")`, that runs a query and returns a pandas DataFrame (a table).

You normally don't need to change the path. If you get "Database not found", the `.db` file isn't in
`data/sample/` yet (see the README).

In [1]:
import sqlite3
from pathlib import Path

import pandas as pd

pd.set_option("display.max_colwidth", 80)  # max characters shown per cell in tables; raise it to see longer text

# Works whether VS Code runs the notebook from notebooks/ or from the project root
ROOT = Path.cwd() if (Path.cwd() / "data" / "sample").exists() else Path.cwd().parent
DB_PATH = ROOT / "data" / "sample" / "agent_skills_sample.db"
assert DB_PATH.exists(), f"Database not found at {DB_PATH} (see README)"

# Read-only, so nothing in the database can be changed by accident
conn = sqlite3.connect(f"{DB_PATH.resolve().as_uri()}?mode=ro", uri=True)


# sql("SELECT ...") -> DataFrame. Use ? placeholders for values: sql("... WHERE name = ?", ("pdf",))
def sql(query, params=()):
    return pd.read_sql_query(query, conn, params=params)


print("Connected to", DB_PATH.relative_to(ROOT))

Connected to data\sample\agent_skills_sample.db


## 2. Database overview

The four tables:

- **`artifacts`**: one row per SKILL.md found on GitHub (a repo + path). Holds the skill's `name`, `description`,
  full text (`content`), folder stats and commit info.
- **`artifact_siblings`**: the other files in a skill's folder (scripts, reference docs, ...). Linked to
  `artifacts` by `repo_full_name` + `artifact_path` = `artifacts.repo_full_name` + `path`.
- **`repos`**: GitHub info for each repository (stars, forks, language, license, ...). Linked by
  `repos.full_name` = `artifacts.repo_full_name`.
- **`mining_runs`**: a log of the GitHub searches used to collect the data. Mostly bookkeeping.

The second table shows `skill_occurrences` (every SKILL.md file found) vs `distinct_skill_files` (unique contents).
There are many more occurrences because the same skill gets copied between repos: forks, skill collections, and
people vendoring popular skills. Copies share a `file_sha`, and only one "representative" copy
(`dedup_primary = 1`) stores the text and supporting files.

The last output lists the columns of one table. Set `table` to another name to see its columns.

In [4]:
tables = sql("SELECT name FROM sqlite_master WHERE type = 'table' AND name NOT LIKE 'sqlite_%' ORDER BY name")
tables["rows"] = [sql(f'SELECT COUNT(*) AS n FROM "{t}"')["n"][0] for t in tables["name"]]
display(tables)

display(sql("""
SELECT COUNT(*)                       AS skill_occurrences,
       SUM(dedup_primary)             AS distinct_skill_files,
       COUNT(DISTINCT repo_full_name) AS repos
FROM artifacts
"""))

# Columns of any table
table = "artifacts"  # <- change me: "artifacts", "artifact_siblings", "repos", "mining_runs"
sql(f'PRAGMA table_info("{table}")')[["name", "type"]]

,name,rows
0,artifact_siblings,47829
1,artifacts,29786
2,mining_runs,7
3,repos,11841


,skill_occurrences,distinct_skill_files,repos
0,29786,13000,11841


,name,type
0,repo_full_name,TEXT
1,path,TEXT
2,filename,TEXT
3,location_class,TEXT
4,file_sha,TEXT
5,discovered_at,TEXT
6,content,TEXT
7,content_fetched,INTEGER
8,frontmatter_valid,INTEGER
9,name,TEXT


## 3. Browse skills

Shows `n` distinct skills (representatives only, so each unique SKILL.md appears once), sorted by repo stars.
Change `n` to show more or fewer rows. The result is saved as `skills`, which section 4 uses.

- `repo`, `path`: where the SKILL.md lives on GitHub
- `name`, `description`: from the skill's frontmatter (the block at the top of the file). Can be empty.
- `stars`: the repo's GitHub stars
- `files`: number of other files in the skill's folder
- `has_scripts`, `has_references`: 1 if the folder has script files (.py, .sh, ...) or docs (.md, .txt, ...)
- `copies`: how many places in the dataset have this exact same SKILL.md (1 means it's unique)

To explore differently, uncomment one of the `-- AND ...` filter lines in the query, or swap the `ORDER BY`
for one of the options in the comment next to it.

In [9]:
n = 15  # <- change me: how many skills to show

skills = sql("""
SELECT a.repo_full_name AS repo, a.path, a.name, a.description, r.stars,
       a.sibling_count AS files, a.has_scripts, a.has_references,
       (SELECT COUNT(*) FROM artifacts c WHERE c.file_sha = a.file_sha) AS copies
FROM artifacts a
JOIN repos r ON r.full_name = a.repo_full_name
WHERE a.dedup_primary = 1                -- one row per distinct skill (keep this)
  -- AND a.has_scripts = 1               -- only skills that ship scripts
  -- AND r.language = 'Python'           -- only repos whose main language is Python
  -- AND a.name LIKE '%pdf%'             -- name contains "pdf"
ORDER BY r.stars DESC, a.repo_full_name, a.path   -- try: copies DESC, a.body_chars DESC, RANDOM()
LIMIT ?
""", (n,))
skills

,repo,path,name,description,stars,files,has_scripts,has_references,copies
0,thedaviddias/Front-End-Checklist,skills/broken-images/SKILL.md,broken-images,"Use when reviewing image assets, markup, and CDN or build transforms related...",73218,1,0,1,1
1,thedaviddias/Front-End-Checklist,skills/javascript-minification/SKILL.md,javascript-minification,"Use when reviewing scripts, client components, bundles, or runtime behavior ...",73218,1,0,1,1
2,thedaviddias/Front-End-Checklist,skills/responsive-size/SKILL.md,responsive-size,"Use when reviewing image assets, markup, and CDN or build transforms related...",73218,1,0,1,1
3,thedaviddias/Front-End-Checklist,skills/structured-data/SKILL.md,structured-data,"Use when auditing metadata, crawlability, structured data, or indexability r...",73218,1,0,1,1
4,ClickHouse/ClickHouse,.claude/skills/update-contrib/SKILL.md,update-contrib,Update a ClickHouse third-party library (contrib submodule) to a new version...,48658,0,0,0,1
5,HKUDS/CLI-Anything,skills/cli-anything-musescore/SKILL.md,cli-anything-musescore,"CLI for music notation — transpose, export PDF/audio/MIDI, extract parts, ma...",45368,0,0,0,5
6,rustfs/rustfs,.agents/skills/rust-code-quality/SKILL.md,rust-code-quality,Enforce Rust-specific code quality rules on every code change. Use before me...,29859,1,0,1,1
7,mukul975/Anthropic-Cybersecurity-Skills,skills/testing-api-for-mass-assignment-vulnerability/SKILL.md,testing-api-for-mass-assignment-vulnerability,Tests APIs for mass assignment (auto-binding) vulnerabilities where clients ...,25639,3,1,1,1
8,EveryInc/compound-engineering-plugin,tests/fixtures/sample-plugin/skills/claude-only-skill/SKILL.md,claude-only-skill,A skill restricted to Claude Code only,23221,0,0,0,5
9,trycua/cua,libs/cua-driver/rust/Skills/cua-driver/SKILL.md,cua-driver,"Drive a native GUI app (macOS, Windows, Linux) via the cua-driver CLI (defau...",19790,7,0,1,1


## 4. View a skill

Prints the full SKILL.md text for one skill, i.e. the instructions an agent would read.

- **Easiest:** set `skill_index` to a row number from the table in section 3 (the number on the far left).
- **From a search:** after running section 5, uncomment the `matches.loc[...]` line and pick a row number there.
- **Any skill:** uncomment the last option and type a `repo` and `path` yourself.

Only use one option at a time (comment out the others). If you pick a copy, the text comes from its representative
and the output says which one. Section 6 shows the files for whatever skill you pick here, through the `skill`
variable.

In [37]:
# Option 1: a row number from the section 3 table
# skill_index = 5  # <- change me
# repo, path = skills.loc[skill_index, ["repo", "path"]]

# Option 2: a row number from the section 5 search results (run section 5 first)
# repo, path = matches.loc[0, ["repo", "path"]]  # <- change the 0

# Option 3: any repo/path, typed directly (copies work too)
repo, path = "0xDarkMatter/claude-mods", "skills/security-ops/SKILL.md"

# Copies don't store text, so read it from the representative row with the same file_sha
skill = sql("""
SELECT rep.repo_full_name AS rep_repo, rep.path AS rep_path, rep.name, rep.content
FROM artifacts a
JOIN artifacts rep ON rep.file_sha = a.file_sha AND rep.dedup_primary = 1
WHERE a.repo_full_name = ? AND a.path = ?
""", (repo, path)).iloc[0]

print(f"{repo} / {path}")
if (skill.rep_repo, skill.rep_path) != (repo, path):
    print(f"(identical copy of {skill.rep_repo} / {skill.rep_path})")
print("-" * 80)
print(skill.content)

0xDarkMatter/claude-mods / skills/security-ops/SKILL.md
--------------------------------------------------------------------------------
---
name: security-ops
description: "Security audit orchestrator - parallel dependency scanning, SAST pattern detection, auth/config review. Dispatches 3 audit agents simultaneously, consolidates into OWASP-mapped severity report. Triggers on: security review, security audit, OWASP, XSS, SQL injection, CSRF, authentication, authorization, secrets management, input validation, secure coding, vulnerability scan, dependency audit."
license: MIT
allowed-tools: "Read Edit Write Bash Glob Grep Agent TaskCreate TaskUpdate"
metadata:
  author: claude-mods
  related-skills: auth-ops, testing-ops, debug-ops, monitoring-ops
---

# Security Operations

Orchestrator for security auditing. Detects project stack inline, dispatches three parallel audit agents (dependency, SAST, auth/config review), consolidates into a severity-ranked OWASP-mapped report.

> Findings 

## 5. Keyword search

Finds distinct skills whose SKILL.md text contains `keyword`. Set it to anything you're curious about,
such as `"curl"`, `"password"`, `"subprocess"`, `"sudo"` or `".env"`, then re-run. The search ignores upper/lower case
for English letters and matches anywhere in the text, so `"key"` also matches `"monkey"`.

- The printed number is how many **distinct** SKILL.md files contain the keyword. Copies aren't counted again.
- `limit` only controls how many of those rows are shown, not how many are counted.
- To read one, go to section 4, use the `matches.loc[...]` option with that row number, and run it.

A match only means the text appears somewhere. Usually it's harmless: docs saying "set your API_KEY in .env",
an example command, a warning. It doesn't mean the skill is insecure, so read it before drawing conclusions.

In [24]:
keyword = "sudo"  # <- change me: any text to look for
limit = 10           # <- change me: max rows to display

where = "dedup_primary = 1 AND content LIKE '%' || ? || '%'"
total = sql(f"SELECT COUNT(*) AS n FROM artifacts WHERE {where}", (keyword,))["n"][0]
print(f"{total} distinct SKILL.md files contain {keyword!r} (showing {min(total, limit)})")

matches = sql(f"""
SELECT repo_full_name AS repo, path, name, description
FROM artifacts
WHERE {where}
ORDER BY repo_full_name, path
LIMIT ?
""", (keyword, limit))
matches

162 distinct SKILL.md files contain 'sudo' (showing 10)


,repo,path,name,description
0,26zl/cybersec-toolkit,.claude/skills/configuring-suricata-for-network-monitoring/SKILL.md,configuring-suricata-for-network-monitoring,"Deploys and configures Suricata IDS/IPS with Emerging Threats rulesets, EVE ..."
1,26zl/cybersec-toolkit,.claude/skills/performing-privilege-escalation-assessment/SKILL.md,performing-privilege-escalation-assessment,Performs privilege escalation assessments on compromised Linux and Windows s...
2,91zgaoge/memoh-X,internal/skills/defaults/docx/SKILL.md,docx,"Comprehensive document creation, editing, and analysis with support for trac..."
3,AIA-11-HN-MIB/MIB-MockInterviewAIBot,.claude/skills/ffmpeg/SKILL.md,ffmpeg,Guide for using FFmpeg - a comprehensive multimedia framework for video/audi...
4,Abulfadl-Ahmadi/Experimental-Physics-Spring-2026,.skills/wiki-query/SKILL.md,wiki-query,Answer questions by searching the compiled Obsidian wiki. Use this skill whe...
5,AdriP-maker/Proyecto_Vaquitas,.agents/skills/ui-ux-pro-max/SKILL.md,ui-ux-pro-max,Comprehensive design guide for web and mobile applications.
6,AlvaroBiano/bianinho-backup-1777760438,skills/autonomous-ai-agents/hermes-agent/SKILL.md,hermes-agent,"Complete guide to using and extending Hermes Agent — CLI usage, setup, confi..."
7,Angelo0218/blocktic-backend,.claude/skills/writing-hookify-rules/SKILL.md,Writing Hookify Rules,"This skill should be used when the user asks to ""create a hookify rule"", ""wr..."
8,Aradotso/hermes-skills,skills/hermes-dec-bytecode-reverse-engineering/SKILL.md,hermes-dec-bytecode-reverse-engineering,Disassemble and decompile React Native Hermes bytecode (HBC) files for rever...
9,ArtMin96/soloist,.claude/skills/tauri-testing/SKILL.md,testing-tauri-apps,Guides developers through testing Tauri applications including unit testing ...


## 6. Supporting files

A skill is a folder: the SKILL.md plus anything else the author added. That could be scripts the agent can run
(`scripts/setup.sh`), reference docs it reads when needed (`references/api.md`), templates or data files.
This cell lists the files next to the skill you picked in **section 4**. To check another skill, change it there
and re-run 4, then this cell.

- `file`: path inside the skill folder, `bytes`: file size
- `text_stored`: 1 if the dataset has the file's text. Binaries (images, etc.), very large files and folders with
  too many files aren't stored, so those show 0.

To read a file, paste a name from the `file` column into `show_file`, e.g. `show_file = "references/rule.md"`.
Many skills have no extra files at all, so an empty table is normal.

In [26]:
# Uses `skill` from section 4
files = sql("""
SELECT entry_name AS file, entry_size AS bytes, content IS NOT NULL AS text_stored
FROM artifact_siblings
WHERE repo_full_name = ? AND artifact_path = ? AND entry_type = 'file'
ORDER BY entry_name
""", (skill.rep_repo, skill.rep_path))
print(f"{len(files)} files in {skill.rep_repo} / {skill.rep_path}")
display(files)

show_file = ""  # <- change me: a name from the "file" column; leave "" to skip
if show_file:
    text = sql("""
    SELECT content FROM artifact_siblings
    WHERE repo_full_name = ? AND artifact_path = ? AND entry_name = ?
    """, (skill.rep_repo, skill.rep_path, show_file))["content"]
    if text.empty:
        print(f"No file named {show_file!r} in this skill's folder (check the spelling, or the skill picked in section 4)")
    else:
        print(text[0] if text[0] is not None else "(text not stored for this file)")

3 files in 26zl/cybersec-toolkit / .claude/skills/configuring-suricata-for-network-monitoring/SKILL.md


,file,bytes,text_stored
0,LICENSE,11255,1
1,references/api-reference.md,2627,1
2,scripts/agent.py,7814,1


## 7. Custom SQL

Write any SQLite query between the triple quotes and run the cell. The result shows up as a table.
The example lists the repos with the most distinct skills. Some other things to try:

```sql
-- first 10 rows of a table
SELECT * FROM repos LIMIT 10
```

```sql
-- distinct skills that ship scripts, from repos with 100+ stars
SELECT a.repo_full_name, a.path, a.name, r.stars
FROM artifacts a JOIN repos r ON r.full_name = a.repo_full_name
WHERE a.dedup_primary = 1 AND a.has_scripts = 1 AND r.stars >= 100
ORDER BY r.stars DESC
```

Tips: keep `dedup_primary = 1` when you want each unique skill counted once, and drop it to count every occurrence.
Section 2 shows a table's column names. Save results with `df = sql(query)` if you want to keep working with them in
section 8.

In [19]:
# <- change me: replace the query with your own
query = """
SELECT r.full_name, r.stars, r.language, COUNT(*) AS distinct_skills
FROM artifacts a
JOIN repos r ON r.full_name = a.repo_full_name
WHERE a.dedup_primary = 1
GROUP BY r.full_name
ORDER BY distinct_skills DESC
LIMIT 20
"""
sql(query)

,full_name,stars,language,distinct_skills
0,MikeCheng1208/BattleTree,2,Vue,669
1,majiayu000/claude-skill-registry-data,16,HTML,297
2,NeuralBlitz/buggy,3,HTML,204
3,Klotzkette/claude-fuer-deutsches-recht,1332,Python,145
4,dvcrn/openclaw-skills-marketplace,23,Python,140
5,diegosouzapw/awesome-omni-skill,48,,137
6,David-Li0406/meta-skill-evloving,0,,132
7,tools-only/X-Skills,7,Python,105
8,NeuralBlitz/ncx,6,,101
9,NeuralBlitz/Agent-Gateway,1,TypeScript,95


## 8. Playground

Free space for your own experiments: pandas on query results, quick checks, whatever. Everything from earlier
cells (`sql`, `skills`, `matches`, `skill`, `files`) is available here. Add more cells below if you need them
(hover under a cell and click **+ Code**).

In [ ]:
# Example (uncomment to try): how long are SKILL.md files, in characters?
# lengths = sql("SELECT name, body_chars FROM artifacts WHERE dedup_primary = 1")
# lengths["body_chars"].describe()


In [ ]:
# Export skills for linting
from pathlib import Path

skills_to_export = [
    ("26zl/cybersec-toolkit", ".claude/skills/implementing-canary-tokens-for-network-intrusion/SKILL.md", "implementing-canary-tokens-for-network-intrusion"),
    ("0xDarkMatter/claude-mods", "skills/security-ops/SKILL.md", "security-ops"),
    ("26zl/cybersec-toolkit", ".claude/skills/configuring-suricata-for-network-monitoring/SKILL.md", "configuring-suricata-for-network-monitoring")]

for repo, path, path_name in skills_to_export:
    result = sql("""
    SELECT content
    FROM artifacts
    WHERE repo_full_name = ?
    AND path = ?
    """, (repo, path)) # ? = placeholders. sql(query, params). Acts like a c++ printf

    if result.empty or result.iloc[0, 0] is None:
        print(f"Skill {path_name}: unavailable")
        continue

    output = Path(f"../exported_skills/{path_name}/SKILL.md") # path() represents file/folder location
    output.parent.mkdir(parents=True, exist_ok=True) # create parent folder, prevent error if folder exists
    output.write_text(result.iloc[0,0], encoding="utf-8") # result.iloc[row, col] retrieves vals stored at that cell
    print(output.resolve())



C:\Users\ziyoo\Documents\year4\SkillGuard\exported_skills\implementing-canary-tokens-for-network-intrusion\SKILL.md
C:\Users\ziyoo\Documents\year4\SkillGuard\exported_skills\security-ops\SKILL.md
C:\Users\ziyoo\Documents\year4\SkillGuard\exported_skills\configuring-suricata-for-network-monitoring\SKILL.md
